# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook simulates parsing synthetic raw EDI lines and Debezium JSON records into structured Bronze output via idempotent `MERGE INTO` operations.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ -q

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, to_json, struct

# Set up Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day03_Batch_CDC") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session initialized with Iceberg catalog.")

In [ ]:
# Re-create the Bronze table if it doesn't exist
create_table_sql = """
CREATE TABLE IF NOT EXISTS local.db.bronze_raw_payload (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
"""

spark.sql("CREATE DATABASE IF NOT EXISTS local.db")
spark.sql(create_table_sql)

In [ ]:
# Simulate X12 Batch ingestion
data_x12 = [("ISA*00*          *00*          *ZZ*1234567890     *ZZ*9876543210     *200101*1200*U*00401*000000001*0*T*:~",)]
df_x12 = spark.createDataFrame(data_x12, ["value"])

bronze_x12_df = df_x12.select(
    current_timestamp().alias("ingestion_timestamp"),
    lit("X12_BATCH").alias("source_system_id"),
    lit("x12_msg_1").alias("payload_id"),
    col("value").alias("raw_payload_json")
)

bronze_x12_df.createOrReplaceTempView("bronze_updates_x12")

# Idempotent Merge
spark.sql("""
MERGE INTO local.db.bronze_raw_payload t
USING bronze_updates_x12 s
ON t.payload_id = s.payload_id
WHEN NOT MATCHED THEN INSERT *
""")

print("X12 batch merged.")

In [ ]:
# Simulate Debezium CDC ingestion
data_cdc = [
    ("c", "{\"id\": 1, \"name\": \"Patient A\"}"),
    ("u", "{\"id\": 2, \"name\": \"Patient B\"}")
]
df_cdc = spark.createDataFrame(data_cdc, ["_cdc_op", "payload"])

bronze_cdc_df = df_cdc.select(
    current_timestamp().alias("ingestion_timestamp"),
    lit("DEBEZIUM_CDC").alias("source_system_id"),
    lit("cdc_msg_seq").alias("payload_id"),
    col("payload").cast("string").alias("raw_payload_json")
)

bronze_cdc_df.createOrReplaceTempView("bronze_updates_cdc")

# Idempotent Merge
spark.sql("""
MERGE INTO local.db.bronze_raw_payload t
USING bronze_updates_cdc s
ON t.raw_payload_json = s.raw_payload_json
WHEN NOT MATCHED THEN INSERT *
""")
print("CDC batch merged.")

In [ ]:
spark.sql("SELECT * FROM local.db.bronze_raw_payload WHERE source_system_id IN ('X12_BATCH', 'DEBEZIUM_CDC')").show(truncate=False)